# Semana 09 — Merge/Join: combinando estruturas de dados

## Notebook Professor

Este notebook contém as **soluções esperadas**, interpretações e pontos de discussão do Notebook Estudante.

A sequência é proposital:

**problema → estratégia → custo → nova representação → comparação → cardinalidade → dados reais**

> **Condução:** antes de mostrar cada solução, peça aos estudantes que registrem sua previsão. A diferença entre previsão e resultado é parte do conteúdo, não apenas uma etapa para chegar à resposta correta.


## Preparação — dados pequenos

Começaremos com duas coleções simples. A matrícula será a informação comum entre elas.


In [ ]:
estudantes = [
    {"matricula": 101, "nome": "Ana"},
    {"matricula": 102, "nome": "Bruno"},
    {"matricula": 103, "nome": "Carla"},
    {"matricula": 104, "nome": "Diego"},
]

notas = [
    {"matricula": 103, "nota": 8.5},
    {"matricula": 101, "nota": 9.0},
    {"matricula": 104, "nota": 7.5},
]

print(estudantes)
print(notas)


# Atividade 1 — Como combinar duas coleções?

Queremos produzir registros contendo:

```text
matrícula | nome | nota
```

Sem usar Pandas, responda antes de programar:

1. Como você procuraria a nota correspondente a cada estudante?
2. Com 4 estudantes e 3 registros de notas, qual é o **máximo** de comparações que sua estratégia poderia realizar?
3. Se tivéssemos `n` estudantes e `m` notas, como o número de comparações cresceria?

### Resposta esperada

**Estratégia:** para cada estudante, percorrer a lista de notas até encontrar uma matrícula correspondente.

**Máximo de comparações para o exemplo:** `4 × 3 = 12`, se cada estudante precisar examinar todos os registros de notas.

**Crescimento esperado:** `n × m`, isto é, `O(nm)`. Quando `n` e `m` têm ordem de grandeza semelhante, podemos interpretar como comportamento quadrático.

> **Condução:** alguns estudantes podem dizer 9 ou outro valor ao considerar interrupções antecipadas. Não tratar imediatamente como erro: separar **comparações desta instância concreta** de **máximo/pior caso da estratégia**.


## Implemente a estratégia mais direta

Complete o código. Não use `dict`, `set`, `merge` ou outra estrutura auxiliar nesta primeira solução.


In [ ]:
resultado = []
comparacoes = 0

for estudante in estudantes:
    for registro_nota in notas:
        # Toda tentativa de verificar se as chaves coincidem conta como comparação.
        comparacoes += 1

        if estudante["matricula"] == registro_nota["matricula"]:
            resultado.append({
                "matricula": estudante["matricula"],
                "nome": estudante["nome"],
                "nota": registro_nota["nota"],
            })

            # É seguro interromper porque, neste exemplo, cada matrícula
            # ocorre no máximo uma vez na coleção de notas.
            break

print("Resultado:", resultado)
print("Comparações:", comparacoes)


### Interpretação

Depois de executar, responda:

- O número observado foi igual ao máximo previsto?
- Se foi menor, por quê?
- Em que situação o `break` é seguro?
- Se a mesma matrícula pudesse aparecer várias vezes em `notas`, o que mudaria?

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


# Atividade 2 — Podemos mudar a representação?

Na Semana 05 vimos que uma chave pode ser usada para organizar uma tabela hash.

A ideia agora é transformar:

```text
lista de notas
```

em algo conceitualmente semelhante a:

```text
matrícula → nota
```

## 2.1 — Construa um índice

Complete:


In [ ]:
indice_notas = {}

for registro in notas:
    matricula = registro["matricula"]

    # A matrícula passa a ser a chave de acesso à nota.
    indice_notas[matricula] = registro["nota"]

print(indice_notas)


## 2.2 — Use o índice para combinar

Agora tente produzir o mesmo resultado da Atividade 1 **sem um segundo `for` aninhado**.


In [ ]:
resultado_hash = []

for estudante in estudantes:
    matricula = estudante["matricula"]

    if matricula in indice_notas:
        resultado_hash.append({
            "matricula": matricula,
            "nome": estudante["nome"],
            "nota": indice_notas[matricula],
        })

resultado_hash


## 2.3 — Onde o custo mudou?

Preencha conceitualmente:

| Questão | Nested-loop | Com índice hash |
|---|---|---|
| Existe preparação? | | |
| O que é repetido para cada estudante? | | |
| Há estrutura auxiliar? | | |
| Custo aproximado | | |

Depois explique:

> **Por que dizer que uma consulta hash é `O(1)` em média não significa dizer que o computador realiza literalmente uma única operação?**

**Resposta esperada:** `O(1)` descreve como o custo cresce em relação ao tamanho da entrada, e não uma contagem literal de instruções. Calcular o hash, localizar uma posição, comparar chaves e eventualmente tratar colisões pode envolver várias operações. No comportamento médio esperado, porém, esse trabalho não cresce proporcionalmente a `n`.

| Questão | Nested-loop | Com índice hash |
|---|---|---|
| Existe preparação? | praticamente não | sim, construir o índice |
| O que é repetido? | busca sequencial nas notas | consulta pela chave |
| Estrutura auxiliar? | não | `dict` / tabela hash |
| Custo aproximado | `O(nm)` | `O(n+m)` médio |

> **Condução:** recuperar explicitamente a discussão da S05 sobre `O(1)`.


# Atividade 3 — A ordenação pode ajudar?

Considere as duas coleções ordenadas pela matrícula.


### Uma observação sobre `lambda`

Neste código, `lambda` cria uma função pequena usada apenas para informar ao `sorted` **qual valor deve ser usado na ordenação**:

```python
lambda x: x["matricula"]
```

Aqui ela equivale a:

```python
def obter_matricula(x):
    return x["matricula"]
```

Assim, `sorted(estudantes, key=lambda x: x["matricula"])` significa: **ordene os estudantes usando a matrícula como chave**.

Não é necessário dominar `lambda` nesta disciplina; ela aparece apenas como uma forma compacta de fornecer a função de ordenação.


In [ ]:
estudantes_ord = sorted(estudantes, key=lambda x: x["matricula"])
notas_ord = sorted(notas, key=lambda x: x["matricula"])

print(estudantes_ord)
print(notas_ord)


Podemos manter dois índices, `i` e `j`, e fazer ambos avançarem sem voltar.

Antes de completar o algoritmo, diga o que deve acontecer quando:

- `mat_estudante == mat_nota`
- `mat_estudante < mat_nota`
- `mat_estudante > mat_nota`

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


In [ ]:
i = 0
j = 0
resultado_ordenado = []

while i < len(estudantes_ord) and j < len(notas_ord):
    estudante = estudantes_ord[i]
    registro_nota = notas_ord[j]

    mat_estudante = estudante["matricula"]
    mat_nota = registro_nota["matricula"]

    print(f"i={i} ({mat_estudante}) | j={j} ({mat_nota})")

    if mat_estudante == mat_nota:
        resultado_ordenado.append({
            "matricula": mat_estudante,
            "nome": estudante["nome"],
            "nota": registro_nota["nota"],
        })
        i += 1
        j += 1

    elif mat_estudante < mat_nota:
        # Como as notas estão ordenadas e a nota atual já é maior,
        # esta matrícula de estudante não aparecerá depois.
        i += 1

    else:
        # A nota atual tem chave menor que o estudante atual.
        # Como estudantes estão ordenados, esta nota não encontrará par depois.
        j += 1

resultado_ordenado


## Interprete o sort-merge

1. Por que nenhum ponteiro precisa voltar?
2. Se os dados **já estavam ordenados**, qual é o custo do percurso?
3. Se precisarmos ordenar as duas coleções antes, que custo adicional aparece?
4. Em que cenário a ordenação da Semana 08 passa a ser um investimento reutilizável?

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


### Gabarito de discussão

1. Os ponteiros não voltam porque as coleções estão ordenadas; uma chave menor que a chave atual do outro lado não poderá reaparecer adiante.
2. Se já estão ordenadas, o percurso é `O(n+m)`.
3. Se não estão, devemos acrescentar os custos de ordenação: aproximadamente `O(n log n + m log m)`.
4. A ordenação torna-se um investimento reutilizável quando várias operações posteriores usam a mesma ordem/chave.

> **Aceitar:** formulações equivalentes que distingam corretamente **custo de preparação** e **custo do percurso**.

# Atividade 4 — Três estratégias, uma decisão

Considere:

- nested-loop;
- hash join;
- sort-merge.

Escolha uma estratégia e justifique para cada cenário:

1. duas coleções pequenas;
2. dados já ordenados pela chave;
3. muitas consultas reutilizando o mesmo índice;
4. memória muito limitada;
5. uma tabela hash já construída.

Não responda apenas com Big-O. Considere também **preparação, memória e estado inicial dos dados**.

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


### Respostas plausíveis

Não há uma única escolha obrigatória em todos os cenários. Espera-se justificativa coerente:

- **coleções pequenas:** nested-loop pode ser aceitável pela simplicidade;
- **dados já ordenados:** sort-merge é particularmente atraente;
- **muitas consultas com o mesmo índice:** hash ganha força porque o custo de preparação é amortizado;
- **memória muito limitada:** deve-se considerar que a estrutura hash ocupa memória adicional; a resposta depende do contexto;
- **hash já construído:** aproveitar o índice tende a ser natural.

> **Condução:** rejeitar respostas do tipo “hash é sempre melhor”. O objetivo é perceber que representação, preparação e contexto mudam a decisão.

# Atividade 5 — Semântica do join

Agora usaremos Pandas.

Bruno não possui registro em `notas`. Observe o que acontece com diferentes tipos de join.


In [ ]:
import pandas as pd

df_estudantes = pd.DataFrame(estudantes)
df_notas = pd.DataFrame(notas)

df_estudantes


Antes de executar, preveja se Bruno aparecerá em cada resultado:

| Tipo | Bruno aparece? |
|---|---|
| inner | |
| left | |
| right | |
| outer | |

> Preencha antes da execução.


In [ ]:
for tipo in ["inner", "left", "right", "outer"]:
    print("\n---", tipo.upper(), "---")
    display(
        df_estudantes.merge(
            df_notas,
            on="matricula",
            how=tipo
        )
    )


### Explique

A diferença entre `inner`, `left`, `right` e `outer` é principalmente uma questão de **velocidade** ou de **semântica do resultado**?

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


### Resposta esperada

A diferença entre `inner`, `left`, `right` e `outer` é principalmente **semântica**: define quais registros sem correspondência devem permanecer no resultado. Desempenho pode variar, mas não é o critério que define esses quatro tipos.

# Atividade 6 — Cardinalidade: antes de confiar no merge

Considere agora estas duas tabelas:


In [ ]:
matriculas = pd.DataFrame({
    "matricula": [101, 102, 103],
    "nome": ["Ana", "Bruno", "Carla"]
})

disciplinas = pd.DataFrame({
    "matricula": [101, 101, 102, 103, 103, 103],
    "disciplina": [
        "Estruturas",
        "Cálculo",
        "Estruturas",
        "Estruturas",
        "Cálculo",
        "Estatística"
    ]
})

display(matriculas)
display(disciplinas)


## 6.1 — Faça o diagnóstico **antes** do merge

Responda:

1. Qual é a chave?
2. Ela deveria ser única em `matriculas`?
3. Ela deveria ser única em `disciplinas`?
4. Qual cardinalidade você espera: 1:1, 1:N, N:1 ou N:N?
5. Há 3 linhas em `matriculas` e 6 em `disciplinas`. Quantas linhas você espera no `inner join`?

> **Sua previsão:** escreva aqui.


### Resposta esperada

1. Chave: `matricula`.
2. Em `matriculas`, sim: o exemplo representa um cadastro com uma linha por estudante.
3. Em `disciplinas`, não: um estudante pode cursar várias disciplinas.
4. Cardinalidade esperada: **1:N** (`one_to_many`).
5. Esperamos **6 linhas** no inner join, pois cada registro de `disciplinas` encontra exatamente um estudante.

> **Ponto central:** 3 linhas à esquerda e 6 à direita não implicam “algo errado”. O tamanho é consequência da cardinalidade.

## 6.2 — Verifique a unicidade

Complete a função.


In [ ]:
def diagnosticar_chave(df, chave, nome):
    total = len(df)
    distintos = df[chave].nunique(dropna=False)
    unica = df[chave].is_unique
    duplicadas = df.duplicated(subset=[chave], keep=False).sum()

    print(nome)
    print("  linhas:", total)
    print("  chaves distintas:", distintos)
    print("  chave única?", unica)
    print("  linhas envolvidas em duplicatas:", duplicadas)

diagnosticar_chave(matriculas, "matricula", "matriculas")
diagnosticar_chave(disciplinas, "matricula", "disciplinas")


## 6.3 — Quais chaves repetem?

Complete:


In [ ]:
frequencias = disciplinas["matricula"].value_counts(dropna=False)

# Somente frequências maiores que 1 representam chaves repetidas.
repetidas = frequencias[frequencias > 1]

repetidas


## 6.4 — Declare sua expectativa ao Pandas

Se sua hipótese é `1:N`, o Pandas pode verificá-la.

Complete o `validate`:


In [ ]:
resultado_cardinalidade = matriculas.merge(
    disciplinas,
    on="matricula",
    how="inner",
    validate="one_to_many"
)

resultado_cardinalidade


## 6.5 — Compare antes e depois

Registre:

- linhas à esquerda;
- linhas à direita;
- linhas no resultado;
- diferença em relação à tabela de matrículas.

Depois explique **por que** o resultado tem esse tamanho.

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


### Interpretação esperada

O resultado tem 6 linhas porque a tabela da direita representa ocorrências de matrícula por disciplina. O merge materializa cada correspondência existente. A tabela de estudantes não foi simplesmente “duplicada”: cada repetição corresponde a uma relação estudante–disciplina.

In [ ]:
print("matriculas:", len(matriculas))
print("disciplinas:", len(disciplinas))
print("resultado:", len(resultado_cardinalidade))


# Atividade 7 — Quando os dois lados repetem

Antes de executar:


### Resposta esperada

A previsão é **4 linhas**: cada uma das 2 ocorrências de `1` em A combina com cada uma das 2 ocorrências em B.

Em geral, para uma mesma chave:

`a` ocorrências à esquerda × `b` ocorrências à direita = `a × b` linhas potenciais.

> **Condução:** este é o momento para enfatizar que N:N pode causar crescimento inesperado e que “o código executou sem erro” não significa “o resultado está correto para o domínio”.

In [ ]:
a = pd.DataFrame({
    "chave": [1, 1],
    "origem_a": ["A1", "A2"]
})

b = pd.DataFrame({
    "chave": [1, 1],
    "origem_b": ["B1", "B2"]
})

display(a)
display(b)


Há 2 ocorrências da chave `1` em A e 2 em B.

**Quantas linhas você prevê para o inner join? Por quê?**

> **Previsão esperada:** ver orientação do professor nesta seção.


In [ ]:
resultado_nn = a.merge(
    b,
    on="chave",
    how="inner",
    validate="many_to_many"
)

resultado_nn


Generalize:

Se uma mesma chave aparece `a` vezes de um lado e `b` vezes do outro, quantas linhas essa chave pode gerar?

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


# Atividade 8 — Olist: cardinalidade em dados reais

No Olist, trabalharemos principalmente com:

```text
customers
    │
    ▼
orders
    │
    ▼
order_items
    │
    ▼
products
```

Para `orders` e `order_items`, registre **antes de carregar/executar**:

1. `order_id` deveria ser único em `orders`?
2. `order_id` deveria ser único em `order_items`?
3. Qual cardinalidade você espera?
4. Depois do `inner join`, espera menos, o mesmo número ou mais linhas que `orders`?

> **Hipótese esperada:** ver orientação do professor nesta seção.


### Hipótese esperada

- `order_id` deve ser único em `orders`;
- pode repetir em `order_items`, pois um pedido pode possuir vários itens;
- a relação esperada é **1:N**;
- o resultado do `inner join` tende a ter mais linhas que `orders` quando pedidos possuem múltiplos itens, além de depender de quais pedidos têm correspondência.

> **Importante:** confirmar empiricamente a unicidade antes do merge. A regra do domínio é uma expectativa; o diagnóstico verifica se os dados observados a respeitam.

### Pré-requisito para o bloco Olist

Este notebook espera:

```text
dados/olist/olist_orders_dataset.csv
dados/olist/olist_order_items_dataset.csv
```

O código verifica a existência dos arquivos antes da leitura, evitando interromper o restante do notebook.

> **Antes da aula:** disponibilizar esses CSVs no repositório/ambiente da disciplina ou orientar a turma sobre o procedimento de obtenção adotado para a Semana 09. O notebook não faz download automático.


In [ ]:
from pathlib import Path

pasta_olist = Path("dados/olist")
arquivo_orders = pasta_olist / "olist_orders_dataset.csv"
arquivo_items = pasta_olist / "olist_order_items_dataset.csv"

if arquivo_orders.exists() and arquivo_items.exists():
    orders = pd.read_csv(arquivo_orders)
    order_items = pd.read_csv(arquivo_items)

    print("orders:", orders.shape)
    print("order_items:", order_items.shape)
else:
    print("Arquivos Olist ainda não disponíveis neste ambiente.")


## Diagnostique antes de combinar

Use as mesmas perguntas da atividade anterior.

Se os arquivos estiverem disponíveis, complete ou reutilize sua função `diagnosticar_chave`.


In [ ]:
if arquivo_orders.exists() and arquivo_items.exists():
    diagnosticar_chave(orders, "order_id", "orders")
    diagnosticar_chave(order_items, "order_id", "order_items")


## Faça o merge com uma hipótese explícita

Escolha o valor correto para `validate`.


In [ ]:
if arquivo_orders.exists() and arquivo_items.exists():
    pedidos_itens = orders.merge(
        order_items,
        on="order_id",
        how="inner",
        validate="one_to_many"
    )

    print("orders:", len(orders))
    print("order_items:", len(order_items))
    print("pedidos_itens:", len(pedidos_itens))


### Interprete

Se `pedidos_itens` tiver mais linhas que `orders`, isso significa necessariamente que o merge está errado?

Explique usando a cardinalidade do relacionamento.

> **Resposta esperada:** o valor observado pode ser menor que 12 porque o `break` interrompe a busca quando a correspondência aparece antes do fim. O `break` só é semanticamente seguro se quisermos no máximo uma correspondência por estudante — por exemplo, porque a chave é única em `notas`. Se uma matrícula puder aparecer várias vezes e todas as correspondências forem relevantes, interromper na primeira perderá dados.


### Resposta esperada

Não. Ter mais linhas que `orders` é compatível com uma relação 1:N: um pedido com três itens precisa aparecer em três linhas da visão pedido–item. O problema surgiria se o crescimento contrariasse a cardinalidade esperada ou se `orders.order_id`, que deveria ser único, apresentasse duplicatas.

# Atividade 9 — Benchmark: faça uma previsão

Vamos comparar:

- nested-loop;
- hash join;
- sort-merge.

Para tamanhos crescentes (`500`, `1.000`, `2.500`, `5.000`, `10.000`), desenhe ou descreva como você espera que o **tempo de execução** cresça.

Também responda:

1. O hash join deve contabilizar a construção do índice?
2. O sort-merge deve contabilizar a ordenação?
3. Devemos testar também o cenário em que os dados já estão ordenados?
4. Se o índice hash for reutilizado dez vezes, como isso muda a interpretação?

> **Previsão:** escreva aqui antes de observar os resultados do benchmark.


### Hipóteses esperadas

- **nested-loop:** crescimento aproximadamente quadrático quando os tamanhos crescem juntos;
- **hash join:** crescimento aproximadamente linear no comportamento médio, incluindo construção + consultas;
- **sort-merge com ordenação incluída:** dominado pelos termos de ordenação;
- **sort-merge com dados previamente ordenados:** percurso aproximadamente linear.

O benchmark deve separar cenários de preparação e reutilização. Caso contrário, podemos comparar coisas diferentes e tirar uma conclusão enganosa.

# Fechamento — o que uma linha esconde?

Considere:

```python
df1.merge(df2, on="id")
```

Escreva pelo menos **cinco perguntas** que agora você sabe fazer antes de confiar no resultado dessa operação.

> 1.  
> 2.  
> 3.  
> 4.  
> 5.  

## Ponte para a próxima semana

Depois de combinar tabelas, teremos estruturas como:

```text
cliente | pedido | produto | categoria | preço
```

A próxima pergunta será:

> **Como agrupar e resumir grandes coleções de registros?**


### Exemplos de perguntas esperadas

1. Qual é a chave de correspondência?
2. Em qual lado a chave deveria ser única?
3. Qual cardinalidade é esperada?
4. Quais registros sem correspondência devem ser preservados?
5. Quantas linhas esperamos antes de executar?
6. Existem duplicatas inesperadas?
7. O resultado observado respeita a hipótese?
8. Que organização/estrutura pode estar sendo usada para localizar correspondências?

Não é necessário que o estudante use exatamente estas frases.